# Global Power Plant Database — Analyse avec NumPy, Pandas & Matplotlib

**Objectif :** appliquer NumPy, Pandas et Matplotlib (+ Seaborn) sur un vrai jeu de données : la **Global Power Plant Database** (World Resources Institute), qui recense des centrales électriques dans le monde entier.

**Plan du notebook :**
1. Import et nettoyage des données
2. Exploration des données (EDA)
3. Analyse statistique par type de combustible
4. Analyse temporelle (séries dans le temps)
5. Visualisations avancées (Matplotlib & Seaborn)
6. Opérations matricielles (corrélation, valeurs propres / vecteurs propres)
7. Synthèse : NumPy + Pandas + Matplotlib ensemble
8. Rapport de synthèse final


## 1. Import et nettoyage des données

On commence par importer les bibliothèques et charger le fichier CSV `global_power_plant_database.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pour des graphiques plus lisibles
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Import du fichier CSV (low_memory=False car certaines colonnes ont des types mélangés)
df = pd.read_csv('global_power_plant_database.csv', low_memory=False)

print("Dimensions du dataset :", df.shape)
df.head()


### Identification des valeurs manquantes

On regarde quelles colonnes ont le plus de valeurs manquantes (`NaN`), pour décider quoi faire avec.

In [ ]:
# Nombre de valeurs manquantes par colonne, triées par ordre décroissant
valeurs_manquantes = df.isnull().sum().sort_values(ascending=False)

# On affiche seulement les colonnes qui ont au moins une valeur manquante
print(valeurs_manquantes[valeurs_manquantes > 0])


**Constat :**
- Les colonnes `other_fuel1/2/3` (combustibles secondaires) sont presque toujours vides → normal, peu de centrales utilisent plusieurs combustibles.
- Les colonnes `generation_gwh_2013` à `2019` (production réelle annuelle) ont énormément de valeurs manquantes (beaucoup de pays ne déclarent pas cette donnée).
- `commissioning_year` (année de mise en service) est manquante pour environ la moitié des centrales.

**Stratégie de nettoyage :**
- On garde uniquement les colonnes utiles à notre analyse (pas besoin de `wepp_id`, `url`, etc.).
- Pour `capacity_mw`, `latitude`, `longitude` : ce sont les colonnes les plus complètes, on les garde telles quelles.
- Pour `commissioning_year` : on ne supprime pas les lignes, on les exclut simplement des analyses temporelles (sinon on perdrait la moitié du dataset).
- Pour `primary_fuel` : aucune valeur manquante, c'est notre variable de catégorie principale.

In [ ]:
# On sélectionne les colonnes utiles pour notre analyse
colonnes_utiles = ['country', 'country_long', 'name', 'capacity_mw', 'latitude', 'longitude',
                    'primary_fuel', 'commissioning_year',
                    'generation_gwh_2017', 'generation_gwh_2018', 'generation_gwh_2019']

df_clean = df[colonnes_utiles].copy()

# Conversion explicite en types numériques avec NumPy / Pandas
# (utile si certaines valeurs étaient lues comme du texte)
colonnes_numeriques = ['capacity_mw', 'latitude', 'longitude', 'commissioning_year',
                        'generation_gwh_2017', 'generation_gwh_2018', 'generation_gwh_2019']

for colonne in colonnes_numeriques:
    df_clean[colonne] = pd.to_numeric(df_clean[colonne], errors='coerce').astype(np.float64)

# On supprime les lignes qui n'ont pas de capacité (notre variable la plus importante)
df_clean = df_clean.dropna(subset=['capacity_mw'])

print("Dimensions après nettoyage :", df_clean.shape)
df_clean.info()


## 2. Exploration des données (EDA)

### Statistiques clés sur les colonnes numériques

In [ ]:
# Statistiques générales avec Pandas (mean, median, std, min, max, etc.)
df_clean[['capacity_mw', 'latitude', 'longitude', 'commissioning_year']].describe()


In [ ]:
# On peut aussi calculer ces statistiques "à la main" avec NumPy, pour la capacité par exemple
capacites = df_clean['capacity_mw'].dropna().to_numpy()

print(f"Moyenne (NumPy)     : {np.mean(capacites):.2f} MW")
print(f"Médiane (NumPy)     : {np.median(capacites):.2f} MW")
print(f"Écart-type (NumPy)  : {np.std(capacites):.2f} MW")
print(f"Min / Max (NumPy)   : {np.min(capacites):.2f} MW / {np.max(capacites):.2f} MW")


**Observation :** la moyenne (≈163 MW) est largement supérieure à la médiane (≈17 MW). Cela veut dire que la distribution est **très asymétrique** : il existe un petit nombre de très grosses centrales (barrages hydroélectriques, centrales nucléaires...) qui tirent la moyenne vers le haut, alors que la majorité des centrales sont en réalité de taille modeste (panneaux solaires, petites installations).

### Répartition des centrales par pays

In [ ]:
# Top 10 des pays avec le plus de centrales recensées
top_pays = df_clean['country_long'].value_counts().head(10)
print(top_pays)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_pays.values, y=top_pays.index, hue=top_pays.index, palette='viridis', legend=False)
plt.title("Top 10 des pays par nombre de centrales électriques")
plt.xlabel("Nombre de centrales")
plt.ylabel("Pays")
plt.tight_layout()
plt.show()


### Répartition des centrales par type de combustible (`primary_fuel`)

In [ ]:
repartition_fuel = df_clean['primary_fuel'].value_counts()
print(repartition_fuel)

plt.figure(figsize=(10, 6))
sns.barplot(x=repartition_fuel.values, y=repartition_fuel.index, hue=repartition_fuel.index,
            palette='crest', legend=False)
plt.title("Nombre de centrales par type de combustible (dans le monde)")
plt.xlabel("Nombre de centrales")
plt.ylabel("Type de combustible")
plt.tight_layout()
plt.show()


**Observation :** le **Solaire** est le type de combustible le plus fréquent en nombre de centrales (beaucoup de petites installations solaires), suivi de l'**Hydroélectrique** et de l'**Éolien**. Mais attention : "le plus de centrales" ne veut pas dire "la plus grosse capacité totale" — un grand barrage hydroélectrique peut produire autant qu'un millier de petites installations solaires. On regarde ça juste après.

## 3. Analyse statistique par type de combustible

### Capacité installée moyenne par type de combustible

In [ ]:
# Statistiques de capacité (mean, median, std) groupées par type de combustible
stats_par_fuel = df_clean.groupby('primary_fuel')['capacity_mw'].agg(['mean', 'median', 'std', 'count'])
stats_par_fuel = stats_par_fuel.sort_values('mean', ascending=False).round(2)
stats_par_fuel


In [ ]:
plt.figure(figsize=(10, 6))
ordre = stats_par_fuel.index
sns.barplot(x=stats_par_fuel['mean'], y=ordre, hue=ordre, palette='flare', legend=False)
plt.title("Capacité moyenne (MW) par type de combustible")
plt.xlabel("Capacité moyenne (MW)")
plt.ylabel("Type de combustible")
plt.tight_layout()
plt.show()


**Observation :** le **Nucléaire** a, de loin, la capacité moyenne par centrale la plus élevée (les réacteurs nucléaires sont énormes), suivi par le Charbon (Coal) et le Pétrole lourd (Petcoke). Le **Solaire** et le **Waste** ont les capacités moyennes les plus faibles : ce sont souvent de petites installations.

### Test d'hypothèse : la capacité moyenne diffère-t-elle entre deux types de combustible ?

On compare le **Charbon (Coal)** et le **Gaz (Gas)**, les deux énergies fossiles les plus représentées dans le dataset.

- **H0 (hypothèse nulle)** : la capacité moyenne des centrales à charbon est égale à celle des centrales à gaz.
- **H1 (hypothèse alternative)** : la capacité moyenne diffère entre les deux.

On calcule un test t de Student pour deux échantillons indépendants, "à la main" avec NumPy.

In [ ]:
# On récupère les capacités (en MW) pour chaque groupe, sans valeurs manquantes
capacite_coal = df_clean[df_clean['primary_fuel'] == 'Coal']['capacity_mw'].dropna().to_numpy()
capacite_gas  = df_clean[df_clean['primary_fuel'] == 'Gas']['capacity_mw'].dropna().to_numpy()

# Moyennes et écarts-types de chaque groupe
moyenne_coal, std_coal, n_coal = np.mean(capacite_coal), np.std(capacite_coal, ddof=1), len(capacite_coal)
moyenne_gas,  std_gas,  n_gas  = np.mean(capacite_gas),  np.std(capacite_gas, ddof=1),  len(capacite_gas)

print(f"Coal : moyenne = {moyenne_coal:.1f} MW, écart-type = {std_coal:.1f}, n = {n_coal}")
print(f"Gas  : moyenne = {moyenne_gas:.1f} MW, écart-type = {std_gas:.1f}, n = {n_gas}")

# Erreur standard de la différence des deux moyennes
erreur_standard = np.sqrt(std_coal**2 / n_coal + std_gas**2 / n_gas)

# Statistique t (test de Welch, qui ne suppose pas des variances égales)
t_stat = (moyenne_coal - moyenne_gas) / erreur_standard

print(f"\nStatistique t : {t_stat:.2f}")


**Conclusion :** une statistique t aussi élevée (très loin de 0, et bien au-delà du seuil habituel de ±1.96 pour un seuil de confiance de 95%) indique que la différence observée entre les deux moyennes est **statistiquement significative** : on **rejette H0**. Les centrales à charbon ont, en moyenne, une capacité installée significativement plus élevée que les centrales à gaz dans ce dataset.

## 4. Analyse temporelle

La colonne `commissioning_year` indique l'année de mise en service de chaque centrale. On peut s'en servir pour étudier l'évolution du mix énergétique mondial dans le temps.

In [ ]:
# On ne garde que les centrales dont on connaît l'année de mise en service
df_temps = df_clean.dropna(subset=['commissioning_year']).copy()

# On arrondit l'année (certaines valeurs ont des décimales) et on la convertit en entier
df_temps['commissioning_year'] = df_temps['commissioning_year'].round().astype(int)

print("Année la plus ancienne :", df_temps['commissioning_year'].min())
print("Année la plus récente  :", df_temps['commissioning_year'].max())
print("Nombre de centrales avec une date connue :", len(df_temps))


In [ ]:
# Nombre de centrales mises en service par décennie
df_temps['decennie'] = (df_temps['commissioning_year'] // 10) * 10

centrales_par_decennie = df_temps.groupby('decennie').size()

plt.figure(figsize=(12, 6))
plt.bar(centrales_par_decennie.index.astype(str), centrales_par_decennie.values,
        width=0.8, color='seagreen')
plt.title("Nombre de centrales mises en service par décennie")
plt.xlabel("Décennie")
plt.ylabel("Nombre de centrales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Observation :** on voit une nette accélération du nombre de nouvelles centrales à partir des années 2000-2010, ce qui correspond au boom mondial des énergies renouvelables (solaire, éolien) et à l'industrialisation rapide de pays comme la Chine.

### Évolution du mix énergétique (types de combustible) dans le temps

On regarde, décennie par décennie, quelle part de chaque type de combustible a été installée.

In [ ]:
# Tableau croisé : nombre de centrales par décennie et par type de combustible
mix_energetique = pd.crosstab(df_temps['decennie'], df_temps['primary_fuel'])

# On ne garde que les principaux types de combustible (les plus fréquents), pour la lisibilité
principaux_fuels = df_clean['primary_fuel'].value_counts().head(6).index
mix_energetique_principal = mix_energetique[principaux_fuels]

# On convertit chaque ligne (décennie) en pourcentage, pour comparer la proportion plutôt que le nombre brut
mix_pourcentage = mix_energetique_principal.div(mix_energetique_principal.sum(axis=1), axis=0) * 100

mix_pourcentage.plot(kind='area', stacked=True, figsize=(12, 6), colormap='tab10', alpha=0.85)
plt.title("Évolution du mix énergétique mondial par décennie (%)")
plt.xlabel("Décennie")
plt.ylabel("Part des nouvelles centrales (%)")
plt.legend(title="Combustible", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


**Observation :** la part de l'hydroélectrique et du gaz était dominante au 20ème siècle. À partir des années 2010, le **solaire** et l'**éolien** prennent une place de plus en plus importante dans les nouvelles installations, ce qui reflète la transition énergétique mondiale vers les renouvelables.

## 5. Visualisations avancées

### Distribution de la capacité par type de combustible (boxplot)

Un boxplot permet de voir d'un coup d'œil la médiane, la dispersion et les valeurs extrêmes (outliers) pour chaque catégorie.

In [ ]:
# On se limite aux types de combustible les plus fréquents pour la lisibilité
df_top_fuels = df_clean[df_clean['primary_fuel'].isin(principaux_fuels)]

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_top_fuels, x='primary_fuel', y='capacity_mw', hue='primary_fuel',
            palette='Set2', legend=False)
plt.yscale('log')  # échelle logarithmique car certaines centrales sont BEAUCOUP plus grosses que d'autres
plt.title("Distribution de la capacité (MW) par type de combustible")
plt.xlabel("Type de combustible")
plt.ylabel("Capacité (MW, échelle log)")
plt.tight_layout()
plt.show()


### Carte géographique de la distribution des centrales

On utilise `latitude` et `longitude` pour visualiser où se trouvent les centrales dans le monde, colorées par type de combustible.

In [ ]:
df_geo = df_clean.dropna(subset=['latitude', 'longitude'])
df_geo_top = df_geo[df_geo['primary_fuel'].isin(principaux_fuels)]

plt.figure(figsize=(14, 7))
sns.scatterplot(data=df_geo_top, x='longitude', y='latitude', hue='primary_fuel',
                 size='capacity_mw', sizes=(5, 200), alpha=0.5, palette='tab10')
plt.title("Distribution géographique des centrales électriques dans le monde")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(title="Combustible", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


**Observation :** on retrouve très nettement les contours des continents juste avec les coordonnées des centrales ! Les zones les plus denses (Europe, Est de la Chine, Côte Est des USA, Inde) correspondent aux régions les plus industrialisées et les plus peuplées du monde.

## 6. Opérations matricielles : corrélations, valeurs propres et vecteurs propres

On construit une **matrice de corrélation** entre 4 variables numériques : `capacity_mw`, `latitude`, `longitude` et `commissioning_year`. Cette matrice est carrée et symétrique, ce qui permet d'en calculer les valeurs propres (eigenvalues) et vecteurs propres (eigenvectors).

In [ ]:
variables = ['capacity_mw', 'latitude', 'longitude', 'commissioning_year']
df_matrice = df_clean[variables].dropna()

# Matrice de corrélation (avec Pandas, puis convertie en tableau NumPy)
matrice_correlation = df_matrice.corr().to_numpy()

print("Matrice de corrélation :")
print(np.round(matrice_correlation, 3))

# Visualisation avec une heatmap Seaborn
plt.figure(figsize=(6, 5))
sns.heatmap(df_matrice.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title("Matrice de corrélation entre variables numériques")
plt.tight_layout()
plt.show()


In [ ]:
# Calcul des valeurs propres (eigenvalues) et vecteurs propres (eigenvectors)
# avec np.linalg.eig, qui ne fonctionne que sur des matrices carrées
valeurs_propres, vecteurs_propres = np.linalg.eig(matrice_correlation)

print("Valeurs propres :", np.round(valeurs_propres, 3))
print("\nVecteurs propres (en colonnes) :")
print(np.round(vecteurs_propres, 3))


**À quoi servent les valeurs propres et vecteurs propres ici ?**

- Chaque **valeur propre** indique la quantité de "variance" (d'information) capturée par la direction correspondante. Une valeur propre élevée veut dire que cette direction résume beaucoup de variation dans les données.
- Chaque **vecteur propre** est une combinaison des 4 variables d'origine, qui définit une nouvelle "direction" dans l'espace des données.
- C'est exactement le principe utilisé par l'**ACP (Analyse en Composantes Principales / PCA)** : on garde les vecteurs propres associés aux plus grandes valeurs propres pour réduire le nombre de variables tout en gardant le maximum d'information.
- Dans notre contexte (centrales électriques), cela pourrait servir, par exemple, à résumer en une seule "dimension composite" la relation entre la taille d'une centrale, sa localisation géographique et son ancienneté — utile si on voulait ensuite faire un clustering de centrales similaires.

On remarque que `capacity_mw` et `longitude` ont une corrélation positive notable (≈0.34) : cela suggère que certaines régions du globe (zones avec une longitude particulière, comme l'Asie de l'Est) concentrent davantage de très grosses centrales.

## 7. Intégration de NumPy, Pandas et Matplotlib

Quelques exemples concrets, tirés de cette analyse, qui montrent comment les 3 bibliothèques se complètent :

| Bibliothèque | Rôle dans ce notebook |
|---|---|
| **NumPy** | Conversion de types (`pd.to_numeric` + `.astype(np.float64)`), calculs statistiques bas niveau (`np.mean`, `np.std`), test d'hypothèse "à la main" (calcul manuel de la statistique t), algèbre linéaire (`np.linalg.eig`) |
| **Pandas** | Chargement et nettoyage du CSV, `groupby()` pour les statistiques par combustible, `pd.crosstab()` pour le mix énergétique, `dropna()` / `to_numeric()` pour le nettoyage |
| **Matplotlib / Seaborn** | Tous les graphiques : barres, aires empilées, boxplots, nuage de points géographique, heatmap de corrélation |

**Exemple de filtrage avancé combinant NumPy et Pandas :**
On peut utiliser un tableau booléen NumPy pour filtrer un DataFrame Pandas de façon très flexible — par exemple, sélectionner toutes les centrales qui sont à la fois "grandes" (capacité au-dessus de la médiane) ET "récentes" (mises en service après 2010) :

In [ ]:
mediane_capacite = np.median(df_clean['capacity_mw'].dropna())

# Création d'un masque booléen avec NumPy (combinaison de 2 conditions avec &)
masque = (df_clean['capacity_mw'].to_numpy() > mediane_capacite) & \
         (df_clean['commissioning_year'].to_numpy() > 2010)

centrales_grandes_recentes = df_clean[masque]

print(f"Nombre de centrales grandes ET récentes : {len(centrales_grandes_recentes)}")
centrales_grandes_recentes[['name', 'country_long', 'primary_fuel', 'capacity_mw', 'commissioning_year']].head()


## 8. 📝 Rapport de synthèse

### Principaux constats

In [ ]:
pays_plus_centrales = df_clean['country_long'].value_counts().idxmax()
fuel_plus_frequent = df_clean['primary_fuel'].value_counts().idxmax()
fuel_plus_grosse_capacite_moyenne = stats_par_fuel['mean'].idxmax()
capacite_totale_mondiale = df_clean['capacity_mw'].sum()

print("Résumé de l'analyse — Global Power Plant Database")
print("-" * 55)
print(f"Nombre total de centrales analysées : {len(df_clean):,}")
print(f"Capacité installée totale (somme)   : {capacite_totale_mondiale:,.0f} MW")
print(f"Pays avec le plus de centrales      : {pays_plus_centrales}")
print(f"Type de combustible le plus fréquent (en nombre) : {fuel_plus_frequent}")
print(f"Type de combustible avec la + grosse capacité moyenne par centrale : {fuel_plus_grosse_capacite_moyenne}")


### Conclusions

1. **Distribution très asymétrique** : la moyenne de capacité (≈163 MW) est bien supérieure à la médiane (≈17 MW). Quelques centrales géantes (barrages, nucléaire) côtoient une multitude de petites installations (solaire notamment).
2. **Le solaire domine en nombre, pas en taille** : c'est le type de combustible le plus représenté dans le dataset, mais ce sont en moyenne de petites installations comparées au nucléaire ou au charbon.
3. **Le test d'hypothèse confirme** une différence statistiquement significative de capacité moyenne entre le charbon et le gaz : les centrales à charbon sont, en moyenne, nettement plus grosses.
4. **Transition énergétique visible dans le temps** : le mix énergétique mondial a beaucoup évolué, avec une montée forte du solaire et de l'éolien dans les nouvelles installations à partir des années 2010.
5. **La géographie compte** : la simple visualisation des coordonnées (latitude/longitude) des centrales dessine déjà la carte des continents et des zones industrialisées du monde.
6. **L'algèbre linéaire (corrélation, valeurs propres)** révèle une relation modérée entre la taille des centrales et leur position géographique (longitude), ce qui ouvrirait la voie à des analyses plus poussées (clustering, PCA) sur un dataset enrichi.

### Limites de l'analyse
- Près de la moitié des centrales n'ont pas d'année de mise en service connue (`commissioning_year` manquante) : l'analyse temporelle ne porte donc que sur la moitié du dataset.
- Les données de production réelle (`generation_gwh_...`) sont très incomplètes (moins de 30% des centrales) ; on a donc privilégié la **capacité installée** (`capacity_mw`), bien plus complète, comme indicateur principal.
